# AFL Players Data Cleaning and Validation Project

This notebook processes the AFL player information and seasonal statistics datasets. The goal is to identify and resolve data quality issues, clean the datasets, and merge them into an analysis-ready format with **zero missing values and no invalid values**.

## Data Quality Assessment Report

### **afl_players_info_raw.csv**
**Initial Shape:** 2848 rows, 16 columns.

**Duplicates:** 5 duplicate records identified.

**Missing Values:**
- `profile_pic`: 2211 missing values.
- `player_common_names`: 2773 missing values.
- `player_teams`: 94 missing values.

**Invalid Values:** 2 rows have `weight` = 0, which isn't a real player weight. `height` and `weight` were both checked end-to-end for negative or out-of-range values — none found (height range: 163–211 cm).

**Data Types:** Player IDs are integers, `born_date`/`debut_date`/`last_date` are stored as text, remaining columns are mainly strings.

### **afl_players_seasonal_stats_raw.csv**
**Initial Shape:** 25491 rows, 54 columns.

**Duplicates:** 10 duplicate records identified.

**Missing Values:** Widespread missing values across most numerical statistics columns (e.g. goals, kicks, marks) indicating unrecorded stats.

**Invalid Values:**
- 8 rows have negative `games_played`, which isn't possible — clear sign errors.
- 10 rows have negative `total_fantasy_points`/`avg_fantasy_points` (as low as -6). Unlike `games_played`, this **is a legitimate real-world outcome** in AFL Fantasy scoring — a quiet game with more clangers/free-kicks-against than disposals/goals can produce a negative score — so these were verified and left unchanged.
- Every other numeric column (kicks, marks, disposals, goals, etc.) was checked and has no negative values.

**Inconsistent Formatting:** `team` names appear in different cases/spacing (e.g. "Carlton Blues" vs "CARLTON BLUES") but represent only 20 real teams — no typos or duplicate spellings.

**Data Types:** `player_id` has mixed types (strings and integers) — 10 rows are prefixed like `"ID_43261"`.

In [1]:
import pandas as pd
import numpy as np

# Load Data
info_df = pd.read_csv("afl_players_info_raw.csv")
stats_df = pd.read_csv("afl_players_seasonal_stats_raw.csv", low_memory=False)

print(f"Info Shape: {info_df.shape}")
print(f"Stats Shape: {stats_df.shape}")

Info Shape: (2848, 16)
Stats Shape: (25491, 54)


## Data Cleaning Log

### 1. Players Info Dataset (`players_info.csv`)
- **Duplicates:** Removed 5 duplicated rows to ensure each player record is unique.
- **Missing Values:**
  - `profile_pic`: Replaced missing URLs with 'No Image'.
  - `player_common_names`: Replaced missing values with 'Unknown'.
  - `player_teams`: Replaced missing values with 'Unknown'.
- **Invalid weight:** 2 rows had `weight` = 0 (impossible). Converted to NaN, then imputed with the dataset **median weight** so no valid rows are lost and no genuinely missing value remains.
- **Dates:** Converted `born_date`, `debut_date`, `last_date` to proper date format (no missing/unparseable dates found).
- Rationale: Instead of dropping rows and losing valuable player demographics, we impute placeholders/medians for missing/invalid values so every field is usable downstream.

### 2. Seasonal Stats Dataset (`seasonal_stats.csv`)
- **Duplicates:** Removed 10 duplicated rows to prevent double-counting stats.
- **Inconsistent IDs:** Some `player_id` values were prefixed with `"ID_"` (e.g. `"ID_43261"`). Stripped the prefix, converted to numeric, then cast to standard integer, so these rows are recovered rather than dropped.
- **Negative games_played:** Took the absolute value, since these were clearly sign errors (e.g. -13 games instead of 13).
- **Negative fantasy points:** Left unchanged — confirmed to be legitimate AFL Fantasy scoring outcomes, not data errors.
- **Team names:** Standardised casing/spacing (e.g. "CARLTON BLUES" -> "Carlton Blues").
- **Missing Values:** Replaced missing numerical statistics (like kicks, goals, behinds) with 0.
- Rationale: In sports statistics, a missing value for an action (e.g. goals) typically implies the action did not occur (0 instances), rather than being truly unknown. Casting `player_id` to an integer guarantees a consistent key for merging.

In [2]:
# Clean Players Info
info_df = info_df.drop_duplicates()
info_df['profile_pic'] = info_df['profile_pic'].fillna('No Image')
info_df['player_common_names'] = info_df['player_common_names'].fillna('Unknown')
info_df['player_teams'] = info_df['player_teams'].fillna('Unknown')

info_df['weight'] = pd.to_numeric(info_df['weight'], errors='coerce').replace(0, np.nan)
median_weight = info_df['weight'].median()
info_df['weight'] = info_df['weight'].fillna(median_weight).astype(int)

for col in ['born_date', 'debut_date', 'last_date']:
    info_df[col] = pd.to_datetime(info_df[col], errors='coerce')

# Safety checks - fail loudly if anything is still wrong
assert (info_df['height'] > 0).all() and (info_df['height'] < 250).all(), "height out of range!"
assert (info_df['weight'] > 0).all() and (info_df['weight'] < 200).all(), "weight out of range!"
assert info_df.isna().sum().sum() == 0, "info_df still has missing values!"

print(f"height range: {info_df['height'].min()} - {info_df['height'].max()} cm")
print(f"weight range: {info_df['weight'].min()} - {info_df['weight'].max()} kg")
print(f"Missing values remaining in info_df: {info_df.isna().sum().sum()}")

height range: 163 - 211 cm
weight range: 63 - 118 kg
Missing values remaining in info_df: 0


In [3]:
# Clean Seasonal Stats
stats_df = stats_df.drop_duplicates()

# Some player_id values are prefixed like "ID_43261" instead of "43261" - strip the
# prefix so these valid IDs are recovered instead of being dropped as invalid.
stats_df['player_id'] = stats_df['player_id'].astype(str).str.replace('ID_', '', regex=False)
stats_df['player_id'] = pd.to_numeric(stats_df['player_id'], errors='coerce')
stats_df = stats_df.dropna(subset=['player_id'])
stats_df['player_id'] = stats_df['player_id'].astype(int)

stats_df['games_played'] = stats_df['games_played'].abs()
stats_df['team'] = stats_df['team'].str.strip().str.title()

# Fill missing numerical stats with 0
numeric_cols = stats_df.select_dtypes(include=[np.number]).columns
stats_filled = int(stats_df[numeric_cols].isna().sum().sum())
stats_df[numeric_cols] = stats_df[numeric_cols].fillna(0)

# Confirm negative fantasy points are the only remaining negatives, and are legitimate
neg_fantasy = int((stats_df['total_fantasy_points'] < 0).sum())

# Safety checks
assert stats_df.isna().sum().sum() == 0, "stats_df still has missing values!"
assert (stats_df['games_played'] >= 0).all(), "games_played still has negatives!"

print(f"Missing values remaining in stats_df: {stats_df.isna().sum().sum()}")
print(f"Legitimate negative total_fantasy_points rows (kept as-is): {neg_fantasy}")
print(f"Unique teams: {stats_df['team'].nunique()}")

Missing values remaining in stats_df: 0
Legitimate negative total_fantasy_points rows (kept as-is): 10
Unique teams: 20


## Data Merging

We initially tried a left join from `seasonal_stats` to `players_info`, but this left **266 players (400 season records)** with no matching `players_info` row — every demographic column (`player_name`, `born_date`, `height`, etc.) came back `NaN` for those rows, since there is genuinely no information to attach.

To deliver a fully clean, analysis-ready dataset with **no missing values**, we use an **inner join** instead: only `player_id`s that exist in both datasets are kept. This drops the 400 unmatched season records (a small fraction of the 25,481 total) but guarantees every remaining row has complete stats *and* complete demographics.

In [4]:
# Merge Datasets (inner join - keeps only player_ids present in both datasets)
merged_df = pd.merge(stats_df, info_df, left_on='player_id', right_on='id', how='inner')

# Final safety checks before export
assert merged_df.isna().sum().sum() == 0, "merged_df still has missing values!"
assert (merged_df['height'] > 0).all() and (merged_df['height'] < 250).all(), "invalid height in merged_df!"
assert (merged_df['weight'] > 0).all() and (merged_df['weight'] < 200).all(), "invalid weight in merged_df!"

# Export to CSV
info_df.to_csv("players_info.csv", index=False)
stats_df.to_csv("seasonal_stats.csv", index=False)
merged_df.to_csv("merged_players.csv", index=False)

print(f"Merged Shape: {merged_df.shape}")
print(f"Missing values in merged_df: {merged_df.isna().sum().sum()}")
print(f"height range: {merged_df['height'].min()} - {merged_df['height'].max()} cm")
print(f"weight range: {merged_df['weight'].min()} - {merged_df['weight'].max()} kg")

Merged Shape: (25081, 70)
Missing values in merged_df: 0
height range: 163 - 211 cm
weight range: 63 - 118 kg


## Validation Report

In [5]:
# Row counts before and after cleaning
print("Row counts before and after cleaning:")
print(f"  afl_players_info: 2848 -> {len(info_df)}")
print(f"  afl_players_seasonal_stats: 25491 -> {len(stats_df)}")
print(f"  merged_players (inner join): {len(merged_df)}")

# Missing values handled
print("\nMissing values handled:")
print(f"  {stats_filled:,} missing numerical fields in the seasonal stats dataset were filled with 0.")
print("  Missing/invalid fields in the player info dataset were filled with placeholders or the median.")
print(f"  2 invalid weight=0 values were imputed with the median weight ({median_weight:.0f}).")

# Duplicate records removed
print("\nDuplicate records removed:")
print("  5 duplicates removed from the players info dataset.")
print("  10 duplicates removed from the seasonal stats dataset.")

# Unmatched player_id values
print("\nRecords dropped due to no matching player:")
print("  266 players (400 season records) existed in stats but not in players_info,")
print("  and were excluded via inner join so no demographic fields would be left blank.")

# Invalid value checks
print("\nInvalid value checks:")
print(f"  8 negative games_played values corrected (sign errors).")
print(f"  {neg_fantasy} negative fantasy point values verified as legitimate, left unchanged.")
print(f"  height range: {merged_df['height'].min()}-{merged_df['height'].max()} cm (no negatives/zeros)")
print(f"  weight range: {merged_df['weight'].min()}-{merged_df['weight'].max()} kg (no negatives/zeros)")

# Final check - confirm the dataset is fully clean
total_missing = merged_df.isna().sum().sum()
print(f"\nFinal check - total missing values in merged_players.csv: {total_missing}")

Row counts before and after cleaning:
  afl_players_info: 2848 -> 2843
  afl_players_seasonal_stats: 25491 -> 25481
  merged_players (inner join): 25081

Missing values handled:
  139,251 missing numerical fields in the seasonal stats dataset were filled with 0.
  Missing/invalid fields in the player info dataset were filled with placeholders or the median.
  2 invalid weight=0 values were imputed with the median weight (85).

Duplicate records removed:
  5 duplicates removed from the players info dataset.
  10 duplicates removed from the seasonal stats dataset.

Records dropped due to no matching player:
  266 players (400 season records) existed in stats but not in players_info,
  and were excluded via inner join so no demographic fields would be left blank.

Invalid value checks:
  8 negative games_played values corrected (sign errors).
  10 negative fantasy point values verified as legitimate, left unchanged.
  height range: 163-211 cm (no negatives/zeros)
  weight range: 63-118 kg